# Trace Count v17: Decreasing Long-Tail Synthetic Needles with RoPE

| 项目 | 正式设置 |
| --- | --- |
| 模型 | 每个 position-encoding x mode 组合独立 random init；4 layers；4 heads；d_model=256；MLP=1024 |
| 任务 | count inserted marker needles under a decreasing long-tail training distribution |
| prompt/count | prompt length 256；count 1-30 |
| 模型数 | RoPE x non-thinking/thinking = 2 |
| 训练目标 | **v10-style completion-only causal next-token loss** |
| 数字 token | trace index 与最终答案共享 `<1>...<30>` |


**训练目标：v10-style completion-only causal language modeling。**

完整 gold prefix 作为 causal context，但 task prefix 与 prompt/haystack 标签为 `-100`。
non-thinking 只监督最终 count 与 `<EOS>`；thinking 从第一个 trace 数字开始监督到 `<EOS>`。

## 1. 先挂载 Google Drive

In [ ]:
from pathlib import Path

DRIVE_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/"
    "Synthetic_CoT_NiaH_Count/colab_results"
)
DRIVE_READY = False
if Path("/content").exists():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    DRIVE_READY = True
    print("Drive ready:", DRIVE_RESULTS_ROOT)
else:
    print("Local runtime: Drive mount skipped")

## 2. Repo 与 Python 环境

In [ ]:
import json
import os
import signal
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
preferred = Path("/content/Synthetic_CoT_NiaH_Count")
candidates = [Path.cwd(), *Path.cwd().parents, preferred]
repo = next((p.resolve() for p in candidates if (p / "pyproject.toml").exists()), None)
if repo is None:
    subprocess.run(["git", "clone", REPO_URL, str(preferred)], check=True)
    repo = preferred
elif Path("/content").exists() and (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=False)
os.chdir(repo)

probe = subprocess.run(
    [sys.executable, "-c", "import numpy,pandas,scipy,matplotlib,seaborn"],
    capture_output=True,
    text=True,
)
if probe.returncode:
    print(probe.stderr[-2000:])
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
            "--force-reinstall", "numpy==1.26.4", "pandas==2.2.3",
            "scipy==1.13.1", "matplotlib==3.8.4", "seaborn==0.13.2",
        ],
        check=True,
    )
    if Path("/content").exists():
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError("Scientific ABI repaired. Restart and rerun all cells.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
    check=True,
)
src_root = (repo / "src").resolve()
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display

print({
    "repo": str(repo),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "tests/test_synthetic_counting_v15_v17.py"],
        check=True,
    )

## 3. Runtime settings

In [ ]:
VERSION = "v17"
PRESET = "main"                  # use "debug" for an end-to-end check
STAGE = "all"                    # train, attention, state, plots, or all
SEED = 1234
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_ROOT = f"runs/synthetic_counting_{VERSION}"
RUN_NAME = f"{VERSION}_{PRESET}_rope_completion_seed{SEED}"
SKIP_COMPLETED = True
COUNT_SAMPLING = "power"       # "power" or "exponential"
POWER_ALPHA = 1.0              # p(n) proportional to n ** (-alpha)
EXPONENTIAL_BETA = 0.15        # p(n) proportional to exp(-beta * (n - 1))
CHECKPOINT_SYNC_ROOT = (
    DRIVE_RESULTS_ROOT / f"{VERSION}_live_checkpoints" if DRIVE_READY else None
)

from synthetic_counting_v11.config import preset_config
PLANNED_CONFIG = preset_config(
    VERSION,
    PRESET,
    seed=SEED,
    device=DEVICE,
    count_sampling=COUNT_SAMPLING, power_alpha=POWER_ALPHA, exponential_beta=EXPONENTIAL_BETA,
)
assert (PLANNED_CONFIG.n_layer, PLANNED_CONFIG.n_head) == (4, 4)
assert (PLANNED_CONFIG.n_embd, PLANNED_CONFIG.n_inner) == (256, 1024)
assert PLANNED_CONFIG.loss_scope == "completion"
assert PLANNED_CONFIG.position_encodings == ("rope",)
assert PLANNED_CONFIG.n_embd // PLANNED_CONFIG.n_head == 64
assert PLANNED_CONFIG.rope_base == 10_000.0
assert PLANNED_CONFIG.precision == "bf16"
if PRESET == "main":
    assert PLANNED_CONFIG.train_steps == 10_000
    assert PLANNED_CONFIG.batch_size == 32
    assert PLANNED_CONFIG.warmup_steps == 200
    assert (PLANNED_CONFIG.adam_beta1, PLANNED_CONFIG.adam_beta2) == (0.9, 0.95)
print({
    "version": VERSION,
    "preset": PRESET,
    "device": DEVICE,
    "model_variants": PLANNED_CONFIG.model_variants,
    "number_of_models": len(PLANNED_CONFIG.model_variants),
    "task_type": PLANNED_CONFIG.task_type,
    "noise_source": PLANNED_CONFIG.noise_source,
    "count_sampling": PLANNED_CONFIG.count_sampling,
    "training_objective": PLANNED_CONFIG.to_dict()["training_objective"],
    "rope_base": PLANNED_CONFIG.rope_base,
    "head_dim": PLANNED_CONFIG.n_embd // PLANNED_CONFIG.n_head,
    "batch_size": PLANNED_CONFIG.batch_size,
    "warmup_steps": PLANNED_CONFIG.warmup_steps,
    "adam_betas": (PLANNED_CONFIG.adam_beta1, PLANNED_CONFIG.adam_beta2),
    "precision": PLANNED_CONFIG.precision,
})

## 4. Run v17

In [ ]:
cmd = [
    sys.executable, "-u", "-m", "synthetic_counting_v17.run_v17",
    "--preset", PRESET,
    "--stage", STAGE,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--out-root", OUT_ROOT,
    "--run-name", RUN_NAME,
]
if SKIP_COMPLETED:
    cmd.append("--skip-completed")
if CHECKPOINT_SYNC_ROOT is not None:
    cmd += ["--checkpoint-sync-root", str(CHECKPOINT_SYNC_ROOT)]
cmd += [
    "--count-sampling", COUNT_SAMPLING,
    "--power-alpha", str(POWER_ALPHA),
    "--exponential-beta", str(EXPONENTIAL_BETA),
]

print(" ".join(cmd), flush=True)
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
captured = []
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
    captured.append(line.rstrip())
returncode = process.wait()
if returncode:
    print("---- Last 200 log lines ----")
    print("\n".join(captured[-200:]))
    raise subprocess.CalledProcessError(returncode, cmd)

RUN_DIR = Path(OUT_ROOT) / RUN_NAME
assert (RUN_DIR / "config.json").exists(), RUN_DIR
print("RUN_DIR =", RUN_DIR.resolve())

## 5. Inspect settings and learning dynamics

- `train_total_loss`：只在 supervised completion suffix 上平均的 next-token CE。
- `tf_final_accuracy`：给定 gold prefix，在最终 count 位置对 count vocabulary 取 argmax。
- `tf_trace_marker_accuracy`：仅用于 thinking；给定 gold trace prefix，预测每个 marker identity。
- `ar_final_accuracy`：从 prompt 后自由生成完整 completion，再检查最终 count；它会暴露 trace 错误传播。
- v17 训练分布递减，但 `eval_by_count.csv` 始终对 1-30 每个 count 等量评估。

In [ ]:
config = json.loads((RUN_DIR / "config.json").read_text(encoding="utf-8"))
display(pd.DataFrame([{
    "version": config["version"],
    "architecture": config["architecture"],
    "task_type": config["task_type"],
    "noise_source": config["noise_source"],
    "position_encodings": ", ".join(config["position_encodings"]),
    "training_objective": config["training_objective"],
    "count_sampling": config["count_sampling_definition"],
}]))

for name in (
    "model_specifications.csv",
    "training_count_distribution.csv",
    "time_to_99.csv",
    "autoregressive_by_bin.csv",
):
    path = RUN_DIR / "tables" / name
    if path.exists() and path.stat().st_size:
        display(Markdown(f"**{name}**"))
        display(pd.read_csv(path))

In [ ]:
for name in (
    "training_count_distribution.png",
    "learning_loss.png",
    "learning_accuracy_by_bin.png",
    "final_accuracy_by_count.png",
):
    path = RUN_DIR / "figures" / name
    if path.exists():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(path)))

## 6. Attention and hidden-state diagnostics

Attention heatmaps are descriptive routing signatures rather than causal proof. Broad score rewards
attention mass spread across prompt needles; raw k-to-k mass is attention from trace index `k` to the
kth target occurrence; trace-marker mass measures final-query readout from the gold trace. State tables
report held-out count probes and PCA of per-count centroids.

In [ ]:
for pattern in (
    "attention_signatures_*.png",
    "targeted_retrieval_by_bin_*.png",
    "state_probe_*.png",
    "state_pca_variance_*.png",
    "state_centroids_*.png",
):
    for path in sorted((RUN_DIR / "figures").glob(pattern)):
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))

## 7. Save the complete result bundle to Google Drive

In [ ]:
import shutil
from datetime import datetime

DRIVE_SAVE_COMPLETED = False
SAVED_RESULT_DIR = None
if DRIVE_READY:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    SAVED_RESULT_DIR = DRIVE_RESULTS_ROOT / f"{RUN_NAME}_{stamp}"
    shutil.copytree(RUN_DIR, SAVED_RESULT_DIR, dirs_exist_ok=True)
    DRIVE_SAVE_COMPLETED = (SAVED_RESULT_DIR / "config.json").exists()
    print("Saved:", SAVED_RESULT_DIR)
    print("Verified:", DRIVE_SAVE_COMPLETED)
else:
    print("Drive unavailable; local result remains at", RUN_DIR.resolve())

## 8. Optional: push notebook/code changes to GitHub

In [ ]:
PUSH_TO_GITHUB = False
COMMIT_MESSAGE = f"Add/update {VERSION} experiment"
if PUSH_TO_GITHUB:
    subprocess.run(["git", "status", "--short"], check=True)
    subprocess.run(
        ["git", "add", "src", "tests", "notebooks", "scripts", "docs", "README.md", "pyproject.toml"],
        check=True,
    )
    subprocess.run(["git", "commit", "-m", COMMIT_MESSAGE], check=False)
    subprocess.run(["git", "push", "origin", "HEAD"], check=True)
else:
    print("PUSH_TO_GITHUB=False; no GitHub write performed")

In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = False
if AUTO_DISCONNECT_AFTER_DRIVE_SAVE and DRIVE_SAVE_COMPLETED and Path("/content").exists():
    from google.colab import runtime
    runtime.unassign()
else:
    print("Runtime kept connected")